# TransDetect-Vid — Google Colab demo

Chạy lần lượt các cell bên dưới để upload một video và tạo video kết quả bằng YOLO11n. Notebook chỉ chứa mã cần thiết cho Colab; mã nguồn và tài liệu nằm ở thư mục gốc của repository.

## 1. Clone repository và chọn thư mục làm việc

In [ ]:
from pathlib import Path
import os
import subprocess

local_repo = Path.cwd()
repo_dir = local_repo if (local_repo / 'requirements.txt').is_file() else Path('/content/TransDetect-Vid')
if not (repo_dir / '.git').exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/dzyuu1612/TransDetect-Vid.git', str(repo_dir)],
        check=True,
    )

os.chdir(repo_dir)
print('Thư mục làm việc:', Path.cwd())
assert (repo_dir / 'requirements.txt').is_file()


## 2. Cài dependencies

Dùng `%pip` để package được cài vào đúng Python kernel đang chạy Colab.

In [ ]:
%pip install -q -r requirements.txt torch torchvision

import torch
print('PyTorch:', torch.__version__)

## 3. Tải model YOLO11n

Ultralytics sẽ tự tải `yolo11n.pt` ở lần chạy đầu tiên.

In [ ]:
import torch
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('Thiết bị suy luận:', device)
print('Các lớp của model:', model.names)

## 4. Chọn video

Trên Google Colab, cell sẽ mở hộp upload. Nếu chạy bằng Jupyter/VS Code ở máy local, hãy điền đường dẫn vào biến `LOCAL_VIDEO_PATH` trước khi chạy cell.

In [ ]:
from pathlib import Path

# Chỉ cần điền biến này khi chạy bằng Jupyter/VS Code local.
# Ví dụ: LOCAL_VIDEO_PATH = r'D:\\Videos\\demo.mp4'
LOCAL_VIDEO_PATH = r''

try:
    from google.colab import files
except ModuleNotFoundError:
    files = None

if files is not None:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Hãy upload đúng một video.')
    input_video = next(iter(uploaded))
else:
    if not LOCAL_VIDEO_PATH.strip():
        raise ValueError(
            "Đang chạy local: hãy điền đường dẫn video vào LOCAL_VIDEO_PATH rồi chạy lại cell."
        )
    input_video = LOCAL_VIDEO_PATH.strip().strip('"')

video_path = Path(input_video)
if video_path.suffix.lower() not in {'.mp4', '.avi', '.mov', '.mkv'}:
    raise ValueError('Định dạng không hỗ trợ. Hãy dùng MP4, AVI, MOV hoặc MKV.')
if not video_path.is_file():
    raise FileNotFoundError(f'Không tìm thấy video: {video_path}')

input_video = str(video_path.resolve())
print('Video đã chọn:', input_video)

## 5. Chạy demo và tải video kết quả

In [ ]:
from pathlib import Path
import subprocess
import sys
from IPython.display import FileLink, display

output_video = Path.cwd() / 'outputs' / 'demo_yolo.mp4'
output_video.parent.mkdir(exist_ok=True)
subprocess.run([
    sys.executable, 'main.py',
    '--input', input_video,
    '--method', 'yolo',
    '--model', 'yolo11n.pt',
    '--conf', '0.25',
    '--iou', '0.45',
    '--max-det', '300',
    '--output', str(output_video),
], check=True)

if not output_video.is_file() or output_video.stat().st_size == 0:
    raise RuntimeError(f'Không tạo được video đầu ra: {output_video}')

print(f'Thư mục làm việc: {Path.cwd()}')
print(f'Đã tạo {output_video} ({output_video.stat().st_size / 1024**2:.2f} MB)')
display(FileLink(str(output_video)))

## 6. So sánh YOLO11n và phương pháp truyền thống

Cell này chạy cả hai phương pháp trên cùng video. Có thể mất nhiều thời gian hơn vì video được đọc hai lần.

In [ ]:
from pathlib import Path
import subprocess
import sys
from IPython.display import FileLink, display

output_dir = Path.cwd() / 'outputs'
output_dir.mkdir(exist_ok=True)
comparison_outputs = {
    'YOLO11n': output_dir / 'demo_yolo.mp4',
    'Truyền thống': output_dir / 'demo_classical.mp4',
}

commands = [
    [sys.executable, 'main.py', '--input', input_video, '--method', 'yolo',
     '--model', 'yolo11n.pt', '--conf', '0.25', '--iou', '0.45',
     '--max-det', '300', '--output', str(comparison_outputs['YOLO11n'])],
    [sys.executable, 'main.py', '--input', input_video, '--method', 'classical',
     '--output', str(comparison_outputs['Truyền thống'])],
]

for command in commands:
    subprocess.run(command, check=True)

for method, output_file in comparison_outputs.items():
    if not output_file.is_file() or output_file.stat().st_size == 0:
        raise RuntimeError(f'Không tạo được output {method}: {output_file}')
    print(f'{method}: {output_file} ({output_file.stat().st_size / 1024**2:.2f} MB)')
    display(FileLink(str(output_file)))